# 1. MRI cohort and manifest preparation

This notebook is part of the reproducible data-preparation pipeline used by the downstream modelling notebooks.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import drive
import re

## 1.1. Mount to Drive and check that all the required directories for manifest exist

In [ ]:
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Define main project paths using the actual Google Drive folder

BASE_DIR = Path("/content/drive/My Drive/adni_mri")

MANIFEST_DIR = BASE_DIR / "manifest"
RAW_DOWNLOADS_DIR = BASE_DIR / "raw_downloads"
QC_DIR = BASE_DIR / "qc"
NIFTI_DIR = BASE_DIR / "nifti"
PROCESSED_DIR = BASE_DIR / "processed"

ORIGINAL_MANIFEST_PATH = MANIFEST_DIR / "mri_download_manifest_1661_augmented_subjects.csv"
RECOVERED_PMCI_PATH = MANIFEST_DIR / "selected_recovered_pMCI_baseline_T1_MRI_with_IMAGE_ID_STD.csv"

print("Base folder exists:", BASE_DIR.exists())
print("Manifest folder exists:", MANIFEST_DIR.exists())
print("Raw downloads folder exists:", RAW_DOWNLOADS_DIR.exists())
print("QC folder exists:", QC_DIR.exists())
print("NIfTI folder exists:", NIFTI_DIR.exists())
print("Processed folder exists:", PROCESSED_DIR.exists())

print("\nOriginal manifest exists:", ORIGINAL_MANIFEST_PATH.exists())
print("Recovered pMCI file exists:", RECOVERED_PMCI_PATH.exists())

print("\nFiles in manifest folder:")
for file in sorted(MANIFEST_DIR.iterdir()):
    print("-", file.name)

print("\nFiles in raw_downloads folder:")
for file in sorted(RAW_DOWNLOADS_DIR.iterdir()):
    print("-", file.name)

## 1.2. Load MRI Manifest Files

Load the original 1228-image manifest and the 82 recovered pMCI MRI records.
Check their shapes and columns before making any changes.

In [ ]:
original_manifest = pd.read_csv(ORIGINAL_MANIFEST_PATH)
recovered_pmci = pd.read_csv(RECOVERED_PMCI_PATH)

print("Original manifest shape:", original_manifest.shape)
print("Recovered pMCI shape:", recovered_pmci.shape)

print("\nOriginal manifest columns:")
print(original_manifest.columns.tolist())

print("\nRecovered pMCI columns:")
print(recovered_pmci.columns.tolist())

print("\nOriginal manifest preview:")
display(original_manifest.head())

print("\nRecovered pMCI preview:")
display(recovered_pmci.head())

## 1.3. Inspect Group Counts and Key IDs

Check labels, subject uniqueness, image ID uniqueness, and whether recovered pMCI subjects already overlap with the original manifest.

In [ ]:
print("Original manifest group counts:")
print(original_manifest["final_group"].value_counts(dropna=False))

print("\nRecovered pMCI group counts:")
print(recovered_pmci["final_group"].value_counts(dropna=False))

print("\nSubject/RID checks:")
print("Original rows:", len(original_manifest))
print("Original unique RIDs:", original_manifest["RID"].nunique())
print("Recovered rows:", len(recovered_pmci))
print("Recovered unique RIDs:", recovered_pmci["RID"].nunique())

print("\nImage ID checks:")
print("Original unique image_id:", original_manifest["image_id"].nunique())
print("Recovered unique Image ID:", recovered_pmci["Image ID"].nunique())
print("Recovered unique IMAGE_ID_STD:", recovered_pmci["IMAGE_ID_STD"].nunique())

overlapping_rids = set(original_manifest["RID"]).intersection(set(recovered_pmci["RID"]))

print("\nRecovered pMCI RIDs already present in original manifest:", len(overlapping_rids))

if len(overlapping_rids) > 0:
    print("Overlapping RIDs:")
    print(sorted(overlapping_rids))

## 1.4. Standardise Manifest Columns

In [ ]:
# Make copies
original_std = original_manifest.copy()
recovered_std = pd.DataFrame()

# Add source label to original manifest
original_std["manifest_source"] = "original_1228_manifest"

# Keep image_id as numeric ADNI image ID
original_std["image_id"] = pd.to_numeric(original_std["image_id"], errors="coerce").astype("Int64")


# Build recovered table using the same column names as original
recovered_std["RID"] = recovered_pmci["RID"]
recovered_std["PTID"] = recovered_pmci["PTID"]
recovered_std["subject_id"] = recovered_pmci["Subject ID"]
recovered_std["final_group"] = recovered_pmci["final_group"]

# These recovered subjects are clinically pMCI, meaning baseline diagnosis was MCI
recovered_std["baseline_diagnosis"] = "MCI"

recovered_std["baseline_phase"] = recovered_pmci["baseline_phase"]
recovered_std["baseline_date"] = pd.to_datetime(recovered_pmci["baseline_date"]).dt.date.astype(str)
recovered_std["study_date"] = pd.to_datetime(recovered_pmci["Study Date"]).dt.date.astype(str)

recovered_std["days_from_baseline_mri"] = recovered_pmci["days_from_baseline"]

# Not available in this recovered MRI selection file
recovered_std["conversion_month"] = np.nan
recovered_std["last_followup_month"] = np.nan

recovered_std["description"] = recovered_pmci["Description"]
recovered_std["is_mprage_like"] = recovered_pmci["is_candidate_t1"]
recovered_std["is_repeat"] = recovered_pmci["is_repeat"]

# Use numeric Image ID from recovered file
recovered_std["image_id"] = pd.to_numeric(recovered_pmci["Image ID"], errors="coerce").astype("Int64")

recovered_std["visit"] = recovered_pmci["Visit"]
recovered_std["phase"] = recovered_pmci["Phase"]
recovered_std["research_group"] = recovered_pmci["Research Group"]

# Scanner details are not available in the recovered selected file
recovered_std["field_strength"] = np.nan
recovered_std["manufacturer_clean"] = np.nan
recovered_std["model_family"] = np.nan

recovered_std["manifest_source"] = "recovered_pMCI_82"


# Match final column order
final_columns = [
    "RID",
    "PTID",
    "subject_id",
    "final_group",
    "baseline_diagnosis",
    "baseline_phase",
    "baseline_date",
    "study_date",
    "days_from_baseline_mri",
    "conversion_month",
    "last_followup_month",
    "description",
    "is_mprage_like",
    "is_repeat",
    "image_id",
    "visit",
    "phase",
    "research_group",
    "field_strength",
    "manufacturer_clean",
    "model_family",
    "manifest_source"
]

original_std = original_std[final_columns]
recovered_std = recovered_std[final_columns]

print("Original standardised shape:", original_std.shape)
print("Recovered standardised shape:", recovered_std.shape)

print("\nOriginal image_id preview:")
display(original_std[["RID", "PTID", "final_group", "image_id", "manifest_source"]].head())

print("\nRecovered image_id preview:")
display(recovered_std[["RID", "PTID", "final_group", "image_id", "manifest_source"]].head())

## 1.5. Merge Original and Recovered MRI Records

Combine the 1228 original MRI records with the 82 recovered pMCI records.
Check total rows, group counts, duplicate subjects, and duplicate image IDs.

In [ ]:
updated_manifest = pd.concat(
    [original_std, recovered_std],
    axis=0,
    ignore_index=True
)

print("Updated manifest shape:", updated_manifest.shape)

print("\nFinal group counts:")
print(updated_manifest["final_group"].value_counts(dropna=False))

print("\nSource counts:")
print(updated_manifest["manifest_source"].value_counts(dropna=False))

print("\nSubject/RID checks:")
print("Total rows:", len(updated_manifest))
print("Unique RIDs:", updated_manifest["RID"].nunique())
print("Duplicate RIDs:", updated_manifest["RID"].duplicated().sum())

print("\nImage ID checks:")
print("Unique image IDs:", updated_manifest["image_id"].nunique())
print("Duplicate image IDs:", updated_manifest["image_id"].duplicated().sum())

print("\nMissing key values:")
print(updated_manifest[["RID", "PTID", "subject_id", "final_group", "image_id"]].isna().sum())

## 1.6. Save Updated Manifest and Expected Image IDs

Save the clean 1310-image manifest and create image ID lists for later inventory checking.

In [ ]:
UPDATED_MANIFEST_PATH = MANIFEST_DIR / "updated_mri_manifest_1310.csv"
UPDATED_IMAGE_IDS_ONE_PER_LINE_PATH = MANIFEST_DIR / "updated_expected_image_ids_one_per_line.txt"
UPDATED_IMAGE_IDS_COMMA_PATH = MANIFEST_DIR / "updated_expected_image_ids_comma_separated.txt"

# Safety checks before saving
assert len(updated_manifest) == 1310, "Unexpected total row count."
assert updated_manifest["RID"].nunique() == 1310, "Duplicate RIDs found."
assert updated_manifest["image_id"].nunique() == 1310, "Duplicate image IDs found."
assert updated_manifest[["RID", "PTID", "subject_id", "final_group", "image_id"]].isna().sum().sum() == 0, "Missing key values found."

# Save updated manifest
updated_manifest.to_csv(UPDATED_MANIFEST_PATH, index=False)

# Save expected image IDs
expected_image_ids = updated_manifest["image_id"].astype(int).astype(str).tolist()

with open(UPDATED_IMAGE_IDS_ONE_PER_LINE_PATH, "w") as f:
    f.write("\n".join(expected_image_ids))

with open(UPDATED_IMAGE_IDS_COMMA_PATH, "w") as f:
    f.write(",".join(expected_image_ids))

print("Saved updated manifest:")
print(UPDATED_MANIFEST_PATH)

print("\nSaved one-per-line image IDs:")
print(UPDATED_IMAGE_IDS_ONE_PER_LINE_PATH)

print("\nSaved comma-separated image IDs:")
print(UPDATED_IMAGE_IDS_COMMA_PATH)

print("\nNumber of expected image IDs:", len(expected_image_ids))
print("First 10 image IDs:", expected_image_ids[:10])

## 1.7. Inspect Raw MRI Zip Files

List the MRI zip files and preview their internal filenames.
This helps me understand how image IDs appear before building the inventory checker.

In [ ]:
import zipfile

zip_paths = sorted(RAW_DOWNLOADS_DIR.glob("*.zip"))

print("Number of zip files found:", len(zip_paths))

for zip_path in zip_paths:
    size_gb = zip_path.stat().st_size / (1024 ** 3)

    print("\n" + "=" * 80)
    print("Zip file:", zip_path.name)
    print(f"Size: {size_gb:.2f} GB")

    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()

    print("Number of files/folders inside:", len(names))
    print("\nFirst 20 entries:")
    for name in names[:20]:
        print("-", name)

## 1.8. Build Zip Image Inventory

Scan the zip files without extracting them.
Extract image IDs from paths such as `I7055` and count DICOM files per image.

In [ ]:
inventory_rows = []

for zip_path in zip_paths:
    print("Scanning:", zip_path.name)

    image_file_counts = {}

    with zipfile.ZipFile(zip_path, "r") as z:
        for name in z.namelist():
            if not name.lower().endswith(".dcm"):
                continue

            matches = re.findall(r"I(\d+)", name)

            if len(matches) == 0:
                continue

            image_id = int(matches[-1])
            image_file_counts[image_id] = image_file_counts.get(image_id, 0) + 1

    for image_id, dcm_count in image_file_counts.items():
        inventory_rows.append({
            "zip_file": zip_path.name,
            "image_id": image_id,
            "dcm_file_count": dcm_count
        })

    print("Unique image IDs found:", len(image_file_counts))

zip_inventory = pd.DataFrame(inventory_rows)

print("\nInventory shape:", zip_inventory.shape)

print("\nUnique image IDs per zip:")
print(zip_inventory.groupby("zip_file")["image_id"].nunique())

print("\nPreview:")
display(zip_inventory.head())

## 1.9. Compare Manifest Against Zip Inventory

Match the 1310 expected manifest image IDs against image IDs found inside the MRI zip files.
Identify present images, missing images, and image IDs appearing in more than one zip.

In [ ]:
# Aggregate zip inventory to one row per image_id
zip_inventory_summary = (
    zip_inventory
    .groupby("image_id")
    .agg(
        found_in_zips=("zip_file", lambda x: ", ".join(sorted(x.unique()))),
        n_zips=("zip_file", "nunique"),
        total_dcm_file_count=("dcm_file_count", "sum")
    )
    .reset_index()
)

# Merge expected manifest with found inventory
inventory_report = updated_manifest.merge(
    zip_inventory_summary,
    on="image_id",
    how="left"
)

inventory_report["image_found"] = inventory_report["found_in_zips"].notna()

print("Expected images in updated manifest:", len(updated_manifest))
print("Unique expected image IDs:", updated_manifest["image_id"].nunique())

print("\nUnique image IDs found in zip files:", zip_inventory_summary["image_id"].nunique())

print("\nImage presence:")
print(inventory_report["image_found"].value_counts(dropna=False))

print("\nMissing images by final_group:")
print(inventory_report.loc[~inventory_report["image_found"], "final_group"].value_counts(dropna=False))

print("\nMissing images by manifest_source:")
print(inventory_report.loc[~inventory_report["image_found"], "manifest_source"].value_counts(dropna=False))

print("\nImage IDs found in more than one zip:")
duplicate_zip_images = zip_inventory_summary[zip_inventory_summary["n_zips"] > 1]
print(len(duplicate_zip_images))

if len(duplicate_zip_images) > 0:
    display(duplicate_zip_images)

print("\nFirst missing rows:")
display(
    inventory_report.loc[
        ~inventory_report["image_found"],
        ["RID", "PTID", "final_group", "image_id", "manifest_source", "description", "visit", "phase"]
    ].head(20)
)

## 1.10. Inspect Missing Expected Images

Create a table of the 38 manifest images not found in the zip files.
Check their labels, phase, visit, description, and save image ID lists for re-download.

In [ ]:
missing_expected_images = inventory_report.loc[
    ~inventory_report["image_found"]
].copy()

print("Missing expected images:", len(missing_expected_images))

print("\nMissing by final_group:")
print(missing_expected_images["final_group"].value_counts(dropna=False))

print("\nMissing by phase:")
print(missing_expected_images["phase"].value_counts(dropna=False))

print("\nMissing by visit:")
print(missing_expected_images["visit"].value_counts(dropna=False))

print("\nMissing by description:")
print(missing_expected_images["description"].value_counts(dropna=False))

print("\nMissing image IDs:")
print(missing_expected_images["image_id"].astype(int).astype(str).tolist())

display(
    missing_expected_images[
        [
            "RID",
            "PTID",
            "subject_id",
            "final_group",
            "image_id",
            "baseline_phase",
            "baseline_date",
            "study_date",
            "days_from_baseline_mri",
            "description",
            "visit",
            "phase",
            "research_group",
            "manifest_source"
        ]
    ].sort_values(["phase", "final_group", "RID"])
)

## 1.11. Identified Issue: Are the Selected MRIs Truly Baseline?

The updated manifest correctly merges the original 1228 MRI records with the 82 recovered pMCI records, giving 1310 unique subjects and 1310 unique image IDs.

However, the zip inventory showed that 38 expected images are missing from the downloaded files. All 38 missing images are ADNI4 MRI records from the original manifest.

This raised a more important QC question: the problem is not simply whether the scan belongs to ADNI4, but whether the selected MRI is actually close to the subject’s clinical baseline date. Some missing ADNI4 scans appear to be years after the clinical baseline, meaning they may not be valid baseline MRI scans for prognosis modelling.

Therefore, before re-downloading missing images or starting preprocessing, we need to check the time gap between each selected MRI scan and the clinical baseline date using `days_from_baseline_mri`.

## 1.12. Check Whether Selected MRIs Are Baseline Scans

Measure how far each selected MRI scan is from the clinical baseline date.
Flag scans that are far from baseline before deciding whether to keep, replace, or exclude them.

In [ ]:
baseline_qc = updated_manifest.copy()

baseline_qc["abs_days_from_baseline_mri"] = baseline_qc["days_from_baseline_mri"].abs()

def baseline_distance_category(days):
    if pd.isna(days):
        return "missing"
    elif days == 0:
        return "exact baseline date"
    elif days <= 30:
        return "within 30 days"
    elif days <= 90:
        return "within 90 days"
    elif days <= 180:
        return "within 180 days"
    elif days <= 365:
        return "within 1 year"
    else:
        return "more than 1 year"

baseline_qc["baseline_distance_category"] = baseline_qc["abs_days_from_baseline_mri"].apply(
    baseline_distance_category
)

print("Days from clinical baseline to selected MRI summary:")
display(baseline_qc["days_from_baseline_mri"].describe())

print("\nAbsolute days from clinical baseline to selected MRI summary:")
display(baseline_qc["abs_days_from_baseline_mri"].describe())

print("\nBaseline distance categories:")
print(baseline_qc["baseline_distance_category"].value_counts(dropna=False))

print("\nBaseline distance categories by final_group:")
display(
    pd.crosstab(
        baseline_qc["final_group"],
        baseline_qc["baseline_distance_category"]
    )
)

print("\nPotentially problematic scans: more than 180 days from clinical baseline")
far_from_baseline = baseline_qc[baseline_qc["abs_days_from_baseline_mri"] > 180].copy()

print("Number of scans >180 days from baseline:", len(far_from_baseline))

display(
    far_from_baseline[
        [
            "RID",
            "PTID",
            "final_group",
            "baseline_phase",
            "baseline_date",
            "study_date",
            "days_from_baseline_mri",
            "abs_days_from_baseline_mri",
            "image_id",
            "description",
            "visit",
            "phase",
            "manifest_source"
        ]
    ].sort_values("abs_days_from_baseline_mri", ascending=False)
)

## 1.13. Evaluate Candidate Baseline MRI Windows

Compare how many subjects remain if baseline MRI selection is restricted to common time windows.
No records are removed yet; this only helps decide the QC threshold.

In [ ]:
thresholds = [30, 60, 90, 180, 365]

threshold_summary = []

for threshold in thresholds:
    temp = baseline_qc[baseline_qc["abs_days_from_baseline_mri"] <= threshold]

    row = {
        "max_abs_days_allowed": threshold,
        "total_kept": len(temp),
        "total_excluded": len(baseline_qc) - len(temp),
        "CN_kept": (temp["final_group"] == "CN").sum(),
        "AD_kept": (temp["final_group"] == "AD").sum(),
        "pMCI_kept": (temp["final_group"] == "pMCI").sum(),
        "sMCI_kept": (temp["final_group"] == "sMCI").sum()
    }

    threshold_summary.append(row)

threshold_summary = pd.DataFrame(threshold_summary)

display(threshold_summary)

## 1.14. Interpretation: Late MRIs Cannot Be Treated as Baseline

The modelling task is not progression from sMCI to pMCI. Instead, pMCI and sMCI are retrospective outcome labels assigned to subjects who were MCI at baseline:

- `pMCI`: baseline MCI subject who later converts to AD within the follow-up window.
- `sMCI`: baseline MCI subject who remains MCI during the follow-up window.

Therefore, the MRI used for prognosis should represent the subject at or near the clinical baseline date.

A late MRI scan should not be treated as baseline simply because the subject remained CN or sMCI. If a subject’s clinical baseline was in 2012 but the selected MRI is from 2025, that MRI contains information from many years after the prediction point. Even if the subject stayed CN or sMCI, the scan reflects future survival/stability and can bias the model.

The only scientifically clean way to use such later scans would be to redefine the prediction index date as the MRI date and rebuild the clinical outcome labels relative to that MRI date. Since the current cohort labels are derived from the original DXSUM baseline, late MRIs should either be replaced with earlier baseline/near-baseline scans or excluded from the baseline prognosis cohort.

## 1.15. Characterise Far-from-Baseline MRI Scans

Break selected MRIs into timing windows and inspect which scans are slightly delayed versus clearly non-baseline.
No records are removed in this step.

In [ ]:
timing_qc = baseline_qc.copy()

def detailed_timing_window(days):
    if pd.isna(days):
        return "missing"
    elif days <= 30:
        return "0-30 days"
    elif days <= 60:
        return "31-60 days"
    elif days <= 90:
        return "61-90 days"
    elif days <= 180:
        return "91-180 days"
    elif days <= 365:
        return "181-365 days"
    else:
        return ">365 days"

timing_qc["timing_window"] = timing_qc["abs_days_from_baseline_mri"].apply(detailed_timing_window)

print("Timing windows overall:")
print(timing_qc["timing_window"].value_counts().sort_index())

print("\nTiming windows by final_group:")
display(pd.crosstab(timing_qc["final_group"], timing_qc["timing_window"]))

print("\nTiming windows by MRI phase:")
display(pd.crosstab(timing_qc["phase"], timing_qc["timing_window"]))

print("\nTiming windows by visit:")
display(pd.crosstab(timing_qc["visit"], timing_qc["timing_window"]))

print("\nClearly non-baseline scans >365 days:")
very_late_scans = timing_qc[timing_qc["abs_days_from_baseline_mri"] > 365].copy()

print("Count:", len(very_late_scans))

display(
    very_late_scans[
        [
            "RID",
            "PTID",
            "final_group",
            "baseline_phase",
            "baseline_date",
            "study_date",
            "days_from_baseline_mri",
            "abs_days_from_baseline_mri",
            "image_id",
            "description",
            "visit",
            "phase",
            "manifest_source"
        ]
    ].sort_values("abs_days_from_baseline_mri", ascending=False)
)

## 1.16. Load Full IDA MRI Metadata

Load the full MRI metadata export containing all available candidate MRI scans.
This will be used to check whether far-from-baseline selected scans can be replaced with better baseline/near-baseline alternatives.

In [ ]:
FULL_MRI_METADATA_PATH = MANIFEST_DIR / "idaSearch_6_20_2026_manufacturer.csv"

print("Full MRI metadata file exists:", FULL_MRI_METADATA_PATH.exists())
print("Path:", FULL_MRI_METADATA_PATH)

full_mri_metadata = pd.read_csv(FULL_MRI_METADATA_PATH)

print("\nFull MRI metadata shape:", full_mri_metadata.shape)

print("\nColumns:")
print(full_mri_metadata.columns.tolist())

print("\nPreview:")
display(full_mri_metadata.head())

## 1.17. Inspect Full MRI Metadata

Check available columns, modality values, scan descriptions, visits, and whether the metadata contains the subjects from the current manifest.

In [ ]:
print("Full MRI metadata shape:", full_mri_metadata.shape)

print("\nColumns:")
print(full_mri_metadata.columns.tolist())

print("\nModality counts:")
print(full_mri_metadata["Modality"].value_counts(dropna=False))

print("\nPhase counts:")
print(full_mri_metadata["Phase"].value_counts(dropna=False))

print("\nTop 20 visits:")
print(full_mri_metadata["Visit"].value_counts(dropna=False).head(20))

print("\nTop 30 descriptions:")
print(full_mri_metadata["Description"].value_counts(dropna=False).head(30))

manifest_subjects = set(updated_manifest["subject_id"].astype(str))
metadata_subjects = set(full_mri_metadata["Subject ID"].astype(str))

print("\nSubjects in updated manifest:", len(manifest_subjects))
print("Subjects in full MRI metadata:", len(metadata_subjects))
print("Manifest subjects found in full MRI metadata:", len(manifest_subjects.intersection(metadata_subjects)))
print("Manifest subjects missing from full MRI metadata:", len(manifest_subjects - metadata_subjects))

## 1.18. Load Recovered pMCI MRI Metadata

Load the additional IDA metadata export for the recovered pMCI subjects.
Combine it with the main full MRI metadata file so MRI re-selection can use all available candidate scans.

In [ ]:
RECOVERED_PMCI_METADATA_PATH = MANIFEST_DIR / "idaSearch_6_21_2026_missing_pMCI.csv"

print("Recovered pMCI metadata file exists:", RECOVERED_PMCI_METADATA_PATH.exists())
print("Path:", RECOVERED_PMCI_METADATA_PATH)

recovered_pmci_metadata = pd.read_csv(RECOVERED_PMCI_METADATA_PATH)

print("\nRecovered pMCI metadata shape:", recovered_pmci_metadata.shape)

print("\nColumns:")
print(recovered_pmci_metadata.columns.tolist())

print("\nPreview:")
display(recovered_pmci_metadata.head())

# Combine main metadata + recovered pMCI metadata
all_mri_metadata = pd.concat(
    [full_mri_metadata, recovered_pmci_metadata],
    axis=0,
    ignore_index=True
)

# Remove exact duplicate rows if any
all_mri_metadata = all_mri_metadata.drop_duplicates()

print("\nCombined MRI metadata shape after dropping exact duplicates:", all_mri_metadata.shape)

manifest_subjects = set(updated_manifest["subject_id"].astype(str))
combined_metadata_subjects = set(all_mri_metadata["Subject ID"].astype(str))

print("\nSubjects in updated manifest:", len(manifest_subjects))
print("Subjects in combined MRI metadata:", len(combined_metadata_subjects))
print("Manifest subjects found in combined MRI metadata:", len(manifest_subjects.intersection(combined_metadata_subjects)))
print("Manifest subjects still missing from combined MRI metadata:", len(manifest_subjects - combined_metadata_subjects))

still_missing_subjects = sorted(manifest_subjects - combined_metadata_subjects)

if len(still_missing_subjects) > 0:
    print("\nStill missing subjects:")
    print(still_missing_subjects)

## 1.19. Check Remaining Metadata Coverage Issue

After combining the main IDA metadata export with the recovered pMCI export, only one manifest subject is still unmatched.
Normalize subject IDs and inspect this subject before MRI re-selection.

In [ ]:
# Normalize subject IDs to avoid hidden spaces or formatting issues
updated_manifest["subject_id_clean"] = updated_manifest["subject_id"].astype(str).str.strip()
all_mri_metadata["subject_id_clean"] = all_mri_metadata["Subject ID"].astype(str).str.strip()

manifest_subjects_clean = set(updated_manifest["subject_id_clean"])
metadata_subjects_clean = set(all_mri_metadata["subject_id_clean"])

still_missing_subjects = sorted(manifest_subjects_clean - metadata_subjects_clean)

print("Manifest subjects:", len(manifest_subjects_clean))
print("Metadata subjects:", len(metadata_subjects_clean))
print("Manifest subjects found in combined metadata:", len(manifest_subjects_clean.intersection(metadata_subjects_clean)))
print("Manifest subjects still missing:", len(still_missing_subjects))

print("\nStill missing subjects:")
print(still_missing_subjects)

if len(still_missing_subjects) > 0:
    display(
        updated_manifest.loc[
            updated_manifest["subject_id_clean"].isin(still_missing_subjects),
            [
                "RID",
                "PTID",
                "subject_id",
                "final_group",
                "baseline_phase",
                "baseline_date",
                "study_date",
                "days_from_baseline_mri",
                "image_id",
                "description",
                "visit",
                "phase",
                "manifest_source"
            ]
        ]
    )

## 1.20. Note on the One Subject Missing from Combined Metadata

After combining the main IDA MRI metadata export with the recovered pMCI metadata export, 1309 out of 1310 manifest subjects were found in the combined metadata.

The only unmatched subject is `020_S_6358`. This is not a major issue at this stage because this subject already has a selected MRI in the current manifest, and that MRI is close to the clinical baseline date:

- `image_id`: 996377
- `study_date`: 2018-05-10
- `baseline_date`: 2018-06-21
- `days_from_baseline_mri`: -42

Therefore, this subject does not need immediate correction. If no better metadata is found later, the existing selected MRI can still be kept because it is within a reasonable near-baseline window.

## 1.21. Build Candidate T1 MRI Table

Join all available MRI metadata to the manifest clinical baseline dates.
Flag likely T1 MRI scans and calculate each candidate scan's distance from clinical baseline.

In [ ]:
# Clinical baseline information from the current manifest
clinical_reference = updated_manifest[
    [
        "RID",
        "PTID",
        "subject_id_clean",
        "final_group",
        "baseline_diagnosis",
        "baseline_phase",
        "baseline_date"
    ]
].copy()

clinical_reference["baseline_date"] = pd.to_datetime(clinical_reference["baseline_date"])

# Prepare MRI metadata
mri_candidates = all_mri_metadata.copy()

mri_candidates["subject_id_clean"] = mri_candidates["Subject ID"].astype(str).str.strip()
mri_candidates["study_date"] = pd.to_datetime(mri_candidates["Study Date"], errors="coerce")
mri_candidates["image_id"] = pd.to_numeric(mri_candidates["Image ID"], errors="coerce").astype("Int64")

mri_candidates["description_clean"] = (
    mri_candidates["Description"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# Keep only metadata rows belonging to subjects in the updated manifest
mri_candidates = mri_candidates.merge(
    clinical_reference,
    on="subject_id_clean",
    how="inner"
)

# Candidate T1 structural MRI descriptions
t1_keywords = [
    "mprage",
    "mp-rage",
    "spgr",
    "fspgr",
    "ir-spgr"
]

exclude_keywords = [
    "calibration",
    "localizer",
    "scout",
    "field map",
    "fieldmap",
    "phantom"
]

mri_candidates["is_candidate_t1"] = mri_candidates["description_clean"].apply(
    lambda x: any(keyword in x for keyword in t1_keywords)
    and not any(keyword in x for keyword in exclude_keywords)
)

mri_candidates["is_repeat"] = mri_candidates["description_clean"].str.contains(
    "repeat",
    case=False,
    na=False
)

mri_candidates["days_from_baseline"] = (
    mri_candidates["study_date"] - mri_candidates["baseline_date"]
).dt.days

mri_candidates["abs_days_from_baseline"] = mri_candidates["days_from_baseline"].abs()

print("All metadata rows for manifest subjects:", len(mri_candidates))
print("Unique subjects covered:", mri_candidates["subject_id_clean"].nunique())

print("\nCandidate T1 rows:", mri_candidates["is_candidate_t1"].sum())
print("Unique subjects with at least one candidate T1:", mri_candidates.loc[mri_candidates["is_candidate_t1"], "subject_id_clean"].nunique())

print("\nCandidate T1 timing summary:")
display(
    mri_candidates.loc[
        mri_candidates["is_candidate_t1"],
        "abs_days_from_baseline"
    ].describe()
)

print("\nCandidate T1 preview:")
display(
    mri_candidates.loc[
        mri_candidates["is_candidate_t1"],
        [
            "RID",
            "PTID",
            "subject_id_clean",
            "final_group",
            "baseline_date",
            "study_date",
            "days_from_baseline",
            "abs_days_from_baseline",
            "image_id",
            "Description",
            "Visit",
            "Phase",
            "Research Group",
            "is_repeat"
        ]
    ].sort_values(["subject_id_clean", "abs_days_from_baseline"]).head(20)
)

## 1.22. Re-evaluation of MRI Selection Relative to Clinical Baseline

The current MRI manifest contains one selected T1-weighted MRI scan per subject. However, the timing QC showed that a subset of selected scans is far from the subject’s clinical baseline date.

This is important because the current cohort is defined around a clinical baseline time point. The MRI used for modelling should therefore correspond to the same baseline period, or at least be close enough to it to represent the subject’s baseline state.

At this stage, records should not be removed immediately. The full IDA MRI metadata contains multiple MRI scans per subject across different visits. Therefore, the next step is to check whether subjects with far-from-baseline selected MRIs have alternative T1-weighted scans closer to their clinical baseline date.

This comparison will determine whether problematic records can be corrected by replacing the selected MRI image, or whether they should be excluded from the baseline MRI cohort.

In [ ]:
baseline_qc = baseline_qc.copy()

baseline_qc["subject_id_clean"] = baseline_qc["subject_id"].astype(str).str.strip()

print("subject_id_clean added to baseline_qc.")
print("baseline_qc shape:", baseline_qc.shape)
print("Unique subjects:", baseline_qc["subject_id_clean"].nunique())

display(
    baseline_qc[
        ["RID", "PTID", "subject_id", "subject_id_clean", "final_group", "image_id"]
    ].head()
)

In [ ]:
# Keep only likely T1 MRI candidates
t1_candidates = mri_candidates[mri_candidates["is_candidate_t1"]].copy()

# Prefer closest scan to clinical baseline.
# If there is a tie, prefer non-repeat scans first.
t1_candidates = t1_candidates.sort_values(
    by=[
        "subject_id_clean",
        "abs_days_from_baseline",
        "is_repeat",
        "study_date"
    ],
    ascending=[True, True, True, True]
)

best_t1_candidate = (
    t1_candidates
    .groupby("subject_id_clean", as_index=False)
    .first()
)

print("Subjects with a best available T1 candidate:", len(best_t1_candidate))

# Prepare current selected manifest for comparison
current_selection = baseline_qc[
    [
        "RID",
        "PTID",
        "subject_id_clean",
        "final_group",
        "baseline_phase",
        "baseline_date",
        "study_date",
        "days_from_baseline_mri",
        "abs_days_from_baseline_mri",
        "image_id",
        "description",
        "visit",
        "phase",
        "manifest_source"
    ]
].copy()

current_selection = current_selection.rename(columns={
    "study_date": "current_study_date",
    "days_from_baseline_mri": "current_days_from_baseline",
    "abs_days_from_baseline_mri": "current_abs_days_from_baseline",
    "image_id": "current_image_id",
    "description": "current_description",
    "visit": "current_visit",
    "phase": "current_phase"
})

best_selection = best_t1_candidate[
    [
        "subject_id_clean",
        "study_date",
        "days_from_baseline",
        "abs_days_from_baseline",
        "image_id",
        "Description",
        "Visit",
        "Phase",
        "Research Group",
        "is_repeat"
    ]
].copy()

best_selection = best_selection.rename(columns={
    "study_date": "best_study_date",
    "days_from_baseline": "best_days_from_baseline",
    "abs_days_from_baseline": "best_abs_days_from_baseline",
    "image_id": "best_image_id",
    "Description": "best_description",
    "Visit": "best_visit",
    "Phase": "best_phase",
    "Research Group": "best_research_group",
    "is_repeat": "best_is_repeat"
})

selection_comparison = current_selection.merge(
    best_selection,
    on="subject_id_clean",
    how="left"
)

selection_comparison["has_best_candidate"] = selection_comparison["best_image_id"].notna()

selection_comparison["current_is_best"] = (
    selection_comparison["current_image_id"].astype("Int64") ==
    selection_comparison["best_image_id"].astype("Int64")
)

selection_comparison["improvement_in_abs_days"] = (
    selection_comparison["current_abs_days_from_baseline"] -
    selection_comparison["best_abs_days_from_baseline"]
)

print("Comparison rows:", len(selection_comparison))

print("\nHas best candidate:")
print(selection_comparison["has_best_candidate"].value_counts(dropna=False))

print("\nCurrent selected MRI is already the best available candidate:")
print(selection_comparison["current_is_best"].value_counts(dropna=False))

print("\nImprovement summary in absolute days:")
display(selection_comparison["improvement_in_abs_days"].describe())

print("\nSubjects where current scan is >365 days but best candidate is <=90 days:")
repairable_strict = selection_comparison[
    (selection_comparison["current_abs_days_from_baseline"] > 365) &
    (selection_comparison["best_abs_days_from_baseline"] <= 90)
].copy()

print(len(repairable_strict))

display(
    repairable_strict[
        [
            "RID",
            "PTID",
            "final_group",
            "baseline_date",
            "current_image_id",
            "current_study_date",
            "current_abs_days_from_baseline",
            "current_description",
            "current_visit",
            "current_phase",
            "best_image_id",
            "best_study_date",
            "best_abs_days_from_baseline",
            "best_description",
            "best_visit",
            "best_phase"
        ]
    ].sort_values("current_abs_days_from_baseline", ascending=False)
)

## 1.23. Compare Current MRI Selection with the Closest Available T1 Scan

The current manifest contains one selected MRI per subject. The full IDA metadata contains multiple MRI scans per subject across different visits.

This step selects the T1 MRI scan closest to each subject’s clinical baseline date and compares it with the scan currently selected in the manifest. The aim is to check whether far-from-baseline selections can be corrected using better available MRI candidates.

In [ ]:
# Keep only likely T1 MRI candidates
t1_candidates = mri_candidates[mri_candidates["is_candidate_t1"]].copy()

# Select the closest T1 scan to clinical baseline for each subject
# If two scans are equally close, prefer non-repeat scans first
t1_candidates = t1_candidates.sort_values(
    by=[
        "subject_id_clean",
        "abs_days_from_baseline",
        "is_repeat",
        "study_date"
    ],
    ascending=[True, True, True, True]
)

best_t1_candidate = (
    t1_candidates
    .groupby("subject_id_clean", as_index=False)
    .first()
)

print("Subjects with closest available T1 candidate:", len(best_t1_candidate))


# Prepare current selected scan from the manifest
current_selection = baseline_qc[
    [
        "RID",
        "PTID",
        "subject_id_clean",
        "final_group",
        "baseline_phase",
        "baseline_date",
        "study_date",
        "days_from_baseline_mri",
        "abs_days_from_baseline_mri",
        "image_id",
        "description",
        "visit",
        "phase",
        "manifest_source"
    ]
].copy()

current_selection = current_selection.rename(columns={
    "study_date": "current_study_date",
    "days_from_baseline_mri": "current_days_from_baseline",
    "abs_days_from_baseline_mri": "current_abs_days_from_baseline",
    "image_id": "current_image_id",
    "description": "current_description",
    "visit": "current_visit",
    "phase": "current_phase"
})


# Prepare closest available T1 scan from full metadata
best_selection = best_t1_candidate[
    [
        "subject_id_clean",
        "study_date",
        "days_from_baseline",
        "abs_days_from_baseline",
        "image_id",
        "Description",
        "Visit",
        "Phase",
        "Research Group",
        "is_repeat"
    ]
].copy()

best_selection = best_selection.rename(columns={
    "study_date": "best_study_date",
    "days_from_baseline": "best_days_from_baseline",
    "abs_days_from_baseline": "best_abs_days_from_baseline",
    "image_id": "best_image_id",
    "Description": "best_description",
    "Visit": "best_visit",
    "Phase": "best_phase",
    "Research Group": "best_research_group",
    "is_repeat": "best_is_repeat"
})


# Compare current manifest selection with closest available candidate
selection_comparison = current_selection.merge(
    best_selection,
    on="subject_id_clean",
    how="left"
)

selection_comparison["has_best_candidate"] = selection_comparison["best_image_id"].notna()

selection_comparison["current_is_best"] = (
    selection_comparison["current_image_id"].astype("Int64") ==
    selection_comparison["best_image_id"].astype("Int64")
)

selection_comparison["improvement_in_abs_days"] = (
    selection_comparison["current_abs_days_from_baseline"] -
    selection_comparison["best_abs_days_from_baseline"]
)

print("\nComparison rows:", len(selection_comparison))

print("\nHas closest T1 candidate:")
print(selection_comparison["has_best_candidate"].value_counts(dropna=False))

print("\nCurrent selected scan is already closest available T1:")
print(selection_comparison["current_is_best"].value_counts(dropna=False))

print("\nImprovement in absolute days if closest T1 is used:")
display(selection_comparison["improvement_in_abs_days"].describe())

print("\nSubjects where current scan is >365 days from baseline, but closest T1 is <=90 days:")
repairable_strict = selection_comparison[
    (selection_comparison["current_abs_days_from_baseline"] > 365) &
    (selection_comparison["best_abs_days_from_baseline"] <= 90)
].copy()

print(len(repairable_strict))

display(
    repairable_strict[
        [
            "RID",
            "PTID",
            "final_group",
            "baseline_date",
            "current_image_id",
            "current_study_date",
            "current_abs_days_from_baseline",
            "current_description",
            "current_visit",
            "current_phase",
            "best_image_id",
            "best_study_date",
            "best_abs_days_from_baseline",
            "best_description",
            "best_visit",
            "best_phase"
        ]
    ].sort_values("current_abs_days_from_baseline", ascending=False)
)

Note: So basically what I selected was already best

Now the problem goes from **"did we accidentally choose the wrong MRI"** to **"which timing window should define the clean baseline MRI cohort?"**

## 1.24. Baseline MRI Window Comparison

The closest available T1 MRI has now been identified for each subject. Since some closest scans are still far from the clinical baseline date, the next step is to compare possible timing thresholds.

This table shows how the cohort would change if the baseline MRI cohort were restricted to scans within 90, 180, or 365 days of the clinical baseline date. No records are removed in this step.

In [ ]:
thresholds = [90, 180, 365]

window_summary_rows = []

for threshold in thresholds:
    kept = baseline_qc[baseline_qc["abs_days_from_baseline_mri"] <= threshold].copy()
    excluded = baseline_qc[baseline_qc["abs_days_from_baseline_mri"] > threshold].copy()

    row = {
        "threshold_days": threshold,
        "total_kept": len(kept),
        "total_excluded": len(excluded),
        "CN_kept": (kept["final_group"] == "CN").sum(),
        "AD_kept": (kept["final_group"] == "AD").sum(),
        "pMCI_kept": (kept["final_group"] == "pMCI").sum(),
        "sMCI_kept": (kept["final_group"] == "sMCI").sum(),
        "CN_excluded": (excluded["final_group"] == "CN").sum(),
        "AD_excluded": (excluded["final_group"] == "AD").sum(),
        "pMCI_excluded": (excluded["final_group"] == "pMCI").sum(),
        "sMCI_excluded": (excluded["final_group"] == "sMCI").sum(),
    }

    window_summary_rows.append(row)

window_summary = pd.DataFrame(window_summary_rows)

display(window_summary)

I'm gonna take 90 days because:


Here are both papers in IEEE format:

[1] M. Mehdipour Ghazi, M. Nielsen, A. Pai, M. Modat, M. J. Cardoso, S. Ourselin, and L. Sørensen, "Robust parametric modeling of Alzheimer's disease progression," *NeuroImage*, vol. 225, p. 117460, Jan. 2021, doi: 10.1016/j.neuroimage.2020.117460.

[2] C. M. Stonnington, C. Chu, S. Klöppel, C. R. Jack Jr., J. Ashburner, and R. S. J. Frackowiak, "Predicting clinical scores from magnetic resonance scans in Alzheimer's disease," *NeuroImage*, vol. 51, no. 4, pp. 1405-1413, Jul. 2010, doi: 10.1016/j.neuroimage.2010.03.051.

A couple of notes on these:
- [1] was originally posted as an arXiv preprint (arXiv:1908.05338) in 2019 but was later peer-reviewed and published in *NeuroImage*; the reference above is to the final published version, which is what you'd normally cite.
- Both are open access ,  [1] via Elsevier's CC BY-NC-ND license and [2] via CC BY 3.0 ,  so you can pull the full PDFs directly if you need to check the exact wording of their visit-matching criteria.

## 1.25. Selection of the Primary Baseline MRI Window

For the primary MRI cohort, I restrict selected T1-weighted MRI scans to those acquired within 90 days of the subject’s clinical baseline date.

This threshold is used to keep the MRI data aligned with the clinical baseline used for label construction. Scans acquired much later than baseline may reflect disease state after the prediction point and are therefore not suitable for the main baseline prognosis cohort.

The 90-day threshold is also consistent with a conservative visit-matching approach used in previous ADNI-based MRI studies. More relaxed thresholds, such as 180 or 365 days, can still be tested later as sensitivity analyses.

In [ ]:
BASELINE_WINDOW_DAYS = 90

clean_manifest_90d = baseline_qc[
    baseline_qc["abs_days_from_baseline_mri"] <= BASELINE_WINDOW_DAYS
].copy()

excluded_timing_90d = baseline_qc[
    baseline_qc["abs_days_from_baseline_mri"] > BASELINE_WINDOW_DAYS
].copy()

clean_manifest_90d["baseline_window_days"] = BASELINE_WINDOW_DAYS
clean_manifest_90d["timing_qc_status"] = "included_within_90_days"

excluded_timing_90d["baseline_window_days"] = BASELINE_WINDOW_DAYS
excluded_timing_90d["timing_qc_status"] = "excluded_outside_90_days"

print("Primary clean manifest shape:", clean_manifest_90d.shape)
print("Excluded by 90-day timing rule:", excluded_timing_90d.shape)

print("\nClean 90-day cohort group counts:")
print(clean_manifest_90d["final_group"].value_counts(dropna=False))

print("\nExcluded group counts:")
print(excluded_timing_90d["final_group"].value_counts(dropna=False))

print("\nClean 90-day cohort subject/image checks:")
print("Rows:", len(clean_manifest_90d))
print("Unique RIDs:", clean_manifest_90d["RID"].nunique())
print("Unique subject IDs:", clean_manifest_90d["subject_id"].nunique())
print("Unique image IDs:", clean_manifest_90d["image_id"].nunique())
print("Duplicate RIDs:", clean_manifest_90d["RID"].duplicated().sum())
print("Duplicate image IDs:", clean_manifest_90d["image_id"].duplicated().sum())

print("\nMaximum absolute days from baseline in clean manifest:")
print(clean_manifest_90d["abs_days_from_baseline_mri"].max())

print("\nClean 90-day cohort by manifest source:")
print(clean_manifest_90d["manifest_source"].value_counts(dropna=False))

## 1.26. Save the 90-Day Baseline MRI Cohort

The primary MRI cohort is restricted to subjects whose selected T1-weighted MRI scan is within 90 days of the clinical baseline date.

This step saves the clean 90-day manifest, the excluded timing records, and the expected image ID lists that will be used for inventory checking and later preprocessing.

In [ ]:
CLEAN_90D_MANIFEST_PATH = MANIFEST_DIR / "clean_mri_manifest_90d_1063.csv"
EXCLUDED_90D_TIMING_PATH = QC_DIR / "excluded_mri_timing_outside_90d_247.csv"

CLEAN_90D_IMAGE_IDS_ONE_PER_LINE_PATH = MANIFEST_DIR / "clean_90d_expected_image_ids_one_per_line.txt"
CLEAN_90D_IMAGE_IDS_COMMA_PATH = MANIFEST_DIR / "clean_90d_expected_image_ids_comma_separated.txt"

# Safety checks before saving
assert len(clean_manifest_90d) == 1063, "Unexpected number of rows in clean 90-day manifest."
assert len(excluded_timing_90d) == 247, "Unexpected number of excluded timing records."

assert clean_manifest_90d["RID"].nunique() == 1063, "Duplicate RIDs found in clean 90-day manifest."
assert clean_manifest_90d["image_id"].nunique() == 1063, "Duplicate image IDs found in clean 90-day manifest."

assert clean_manifest_90d["abs_days_from_baseline_mri"].max() <= 90, "Some scans are outside the 90-day window."

# Save manifest and excluded timing records
clean_manifest_90d.to_csv(CLEAN_90D_MANIFEST_PATH, index=False)
excluded_timing_90d.to_csv(EXCLUDED_90D_TIMING_PATH, index=False)

# Save expected image IDs for the clean 90-day cohort
clean_90d_image_ids = clean_manifest_90d["image_id"].astype(int).astype(str).tolist()

with open(CLEAN_90D_IMAGE_IDS_ONE_PER_LINE_PATH, "w") as f:
    f.write("\n".join(clean_90d_image_ids))

with open(CLEAN_90D_IMAGE_IDS_COMMA_PATH, "w") as f:
    f.write(",".join(clean_90d_image_ids))

print("Saved clean 90-day manifest:")
print(CLEAN_90D_MANIFEST_PATH)

print("\nSaved excluded timing records:")
print(EXCLUDED_90D_TIMING_PATH)

print("\nSaved one-per-line image IDs:")
print(CLEAN_90D_IMAGE_IDS_ONE_PER_LINE_PATH)

print("\nSaved comma-separated image IDs:")
print(CLEAN_90D_IMAGE_IDS_COMMA_PATH)

print("\nClean 90-day cohort group counts:")
print(clean_manifest_90d["final_group"].value_counts())

print("\nNumber of expected clean 90-day image IDs:", len(clean_90d_image_ids))
print("First 10 image IDs:", clean_90d_image_ids[:10])

## 1.27. Inventory Check for the 90-Day MRI Cohort

The clean 90-day manifest defines the MRI images that should be used for the primary baseline cohort.

This step checks whether every expected image ID from the 90-day manifest is physically present inside the downloaded MRI zip files. The goal is to confirm that the selected cohort is not only valid in the manifest, but also available for preprocessing.

In [ ]:
# Match clean 90-day manifest against the zip inventory

clean_90d_inventory_report = clean_manifest_90d.merge(
    zip_inventory_summary,
    on="image_id",
    how="left"
)

clean_90d_inventory_report["image_found"] = clean_90d_inventory_report["found_in_zips"].notna()

print("Expected images in clean 90-day manifest:", len(clean_manifest_90d))
print("Unique expected image IDs:", clean_manifest_90d["image_id"].nunique())

print("\nImage presence in downloaded zip files:")
print(clean_90d_inventory_report["image_found"].value_counts(dropna=False))

print("\nMissing images by final_group:")
print(
    clean_90d_inventory_report
    .loc[~clean_90d_inventory_report["image_found"], "final_group"]
    .value_counts(dropna=False)
)

print("\nMissing images by manifest_source:")
print(
    clean_90d_inventory_report
    .loc[~clean_90d_inventory_report["image_found"], "manifest_source"]
    .value_counts(dropna=False)
)

print("\nImages found in more than one zip:")
duplicated_zip_presence = clean_90d_inventory_report[
    clean_90d_inventory_report["n_zips"] > 1
].copy()

print(len(duplicated_zip_presence))

if len(duplicated_zip_presence) > 0:
    display(
        duplicated_zip_presence[
            ["RID", "PTID", "final_group", "image_id", "found_in_zips", "n_zips"]
        ]
    )

print("\nFirst missing rows, if any:")
display(
    clean_90d_inventory_report.loc[
        ~clean_90d_inventory_report["image_found"],
        [
            "RID",
            "PTID",
            "subject_id",
            "final_group",
            "image_id",
            "baseline_date",
            "study_date",
            "days_from_baseline_mri",
            "description",
            "visit",
            "phase",
            "manifest_source"
        ]
    ].head(20)
)

## 1.28. Save Inventory Report for the 90-Day MRI Cohort

All image IDs in the clean 90-day manifest were found inside the downloaded MRI zip files.

This step saves the inventory report for documentation and creates a QC flags table. Since no images are missing and no image IDs appear in multiple zip files, the QC flags file should be empty or contain no serious issues.

In [ ]:
CLEAN_90D_INVENTORY_REPORT_PATH = QC_DIR / "clean_90d_inventory_report.csv"
CLEAN_90D_QC_FLAGS_PATH = QC_DIR / "clean_90d_qc_flags.csv"

# Save full inventory report
clean_90d_inventory_report.to_csv(CLEAN_90D_INVENTORY_REPORT_PATH, index=False)

# Create QC flags
qc_flags = []

missing_images = clean_90d_inventory_report[
    ~clean_90d_inventory_report["image_found"]
].copy()

for _, row in missing_images.iterrows():
    qc_flags.append({
        "RID": row["RID"],
        "PTID": row["PTID"],
        "subject_id": row["subject_id"],
        "final_group": row["final_group"],
        "image_id": row["image_id"],
        "qc_issue": "missing_image_in_zip_files",
        "details": "Expected image ID was not found in the downloaded MRI zip files."
    })

duplicate_zip_images = clean_90d_inventory_report[
    clean_90d_inventory_report["n_zips"] > 1
].copy()

for _, row in duplicate_zip_images.iterrows():
    qc_flags.append({
        "RID": row["RID"],
        "PTID": row["PTID"],
        "subject_id": row["subject_id"],
        "final_group": row["final_group"],
        "image_id": row["image_id"],
        "qc_issue": "image_found_in_multiple_zips",
        "details": f"Image found in multiple zip files: {row['found_in_zips']}"
    })

clean_90d_qc_flags = pd.DataFrame(qc_flags)

clean_90d_qc_flags.to_csv(CLEAN_90D_QC_FLAGS_PATH, index=False)

print("Saved clean 90-day inventory report:")
print(CLEAN_90D_INVENTORY_REPORT_PATH)

print("\nSaved clean 90-day QC flags:")
print(CLEAN_90D_QC_FLAGS_PATH)

print("\nQC flags found:", len(clean_90d_qc_flags))

if len(clean_90d_qc_flags) > 0:
    display(clean_90d_qc_flags)
else:
    print("No missing images or duplicate zip-location issues found.")

## 1.29. Create Preprocessing Extraction Manifest

The 90-day MRI cohort is complete: all 1063 expected image IDs were found in the downloaded zip files.

This step creates a simplified extraction manifest for the next stage. It keeps the subject label, selected image ID, timing information, and the zip file where the DICOM files are located.

In [ ]:
CLEAN_90D_EXTRACTION_MANIFEST_PATH = MANIFEST_DIR / "clean_90d_extraction_manifest_1063.csv"

clean_90d_extraction_manifest = clean_90d_inventory_report[
    [
        "RID",
        "PTID",
        "subject_id",
        "final_group",
        "baseline_diagnosis",
        "baseline_phase",
        "baseline_date",
        "study_date",
        "days_from_baseline_mri",
        "abs_days_from_baseline_mri",
        "image_id",
        "description",
        "visit",
        "phase",
        "research_group",
        "field_strength",
        "manufacturer_clean",
        "model_family",
        "manifest_source",
        "found_in_zips",
        "total_dcm_file_count",
        "image_found"
    ]
].copy()

# Since we already confirmed no image appears in multiple zips,
# rename found_in_zips to source_zip for preprocessing clarity.
clean_90d_extraction_manifest = clean_90d_extraction_manifest.rename(
    columns={"found_in_zips": "source_zip"}
)

# Safety checks
assert len(clean_90d_extraction_manifest) == 1063, "Unexpected extraction manifest row count."
assert clean_90d_extraction_manifest["image_id"].nunique() == 1063, "Duplicate image IDs found."
assert clean_90d_extraction_manifest["RID"].nunique() == 1063, "Duplicate RIDs found."
assert clean_90d_extraction_manifest["image_found"].all(), "Some images are not found in zip files."
assert clean_90d_extraction_manifest["source_zip"].isna().sum() == 0, "Some images have no source zip."

clean_90d_extraction_manifest.to_csv(CLEAN_90D_EXTRACTION_MANIFEST_PATH, index=False)

print("Saved preprocessing extraction manifest:")
print(CLEAN_90D_EXTRACTION_MANIFEST_PATH)

print("\nExtraction manifest shape:", clean_90d_extraction_manifest.shape)

print("\nImages by source zip:")
print(clean_90d_extraction_manifest["source_zip"].value_counts())

print("\nImages by final_group:")
print(clean_90d_extraction_manifest["final_group"].value_counts())

display(clean_90d_extraction_manifest.head())

## 1.30. Check DICOM Series File Counts

Each selected MRI image ID corresponds to a DICOM series inside the downloaded zip files. Before extracting files, the number of DICOM files per selected image should be inspected.

This step does not remove any records. It only checks whether any selected MRI series has an unusually low or unusual DICOM file count, which could indicate an incomplete download or an unexpected scan structure.

In [ ]:
print("DICOM file count summary for selected 90-day MRI series:")
display(clean_90d_extraction_manifest["total_dcm_file_count"].describe())

print("\nMost common DICOM file counts:")
print(clean_90d_extraction_manifest["total_dcm_file_count"].value_counts().head(20))

print("\nDICOM file count by source zip:")
display(
    clean_90d_extraction_manifest
    .groupby("source_zip")["total_dcm_file_count"]
    .describe()
)

print("\nPotentially small DICOM series: fewer than 100 DICOM files")
small_dcm_series = clean_90d_extraction_manifest[
    clean_90d_extraction_manifest["total_dcm_file_count"] < 100
].copy()

print("Count:", len(small_dcm_series))

if len(small_dcm_series) > 0:
    display(
        small_dcm_series[
            [
                "RID",
                "PTID",
                "subject_id",
                "final_group",
                "image_id",
                "source_zip",
                "total_dcm_file_count",
                "description",
                "visit",
                "phase"
            ]
        ].sort_values("total_dcm_file_count")
    )
else:
    print("No selected MRI series has fewer than 100 DICOM files.")

27 selected image IDs only have 1 DICOM file each inside the zip inventory.

That could mean one of two things:

1. **Potential problem:** those MRI series are incomplete or not extracted/downloaded properly.
2. **Less likely but possible:** those are enhanced/multiframe DICOMs where one .dcm file contains the whole volume.

Before worrying, we need to inspect those 27 cases. The important question is: are these real full MRI volumes stored as one file, or incomplete downloads?

## 1.31. Inspect Selected MRI Series with Only One DICOM File

Most selected MRI series contain around 160-211 DICOM files, which is expected for T1-weighted MRI scans stored as slice-based DICOM series.

However, 27 selected image IDs have only one DICOM file. These records require inspection before preprocessing because they may represent incomplete downloads or a different DICOM storage format.

In [ ]:
one_dcm_series = clean_90d_extraction_manifest[
    clean_90d_extraction_manifest["total_dcm_file_count"] == 1
].copy()

print("Selected MRI series with exactly 1 DICOM file:", len(one_dcm_series))

print("\nBy final group:")
print(one_dcm_series["final_group"].value_counts(dropna=False))

print("\nBy source zip:")
print(one_dcm_series["source_zip"].value_counts(dropna=False))

print("\nBy phase:")
print(one_dcm_series["phase"].value_counts(dropna=False))

print("\nBy description:")
print(one_dcm_series["description"].value_counts(dropna=False))

display(
    one_dcm_series[
        [
            "RID",
            "PTID",
            "subject_id",
            "final_group",
            "image_id",
            "source_zip",
            "total_dcm_file_count",
            "description",
            "visit",
            "phase",
            "baseline_date",
            "study_date",
            "days_from_baseline_mri"
        ]
    ].sort_values(["source_zip", "final_group", "RID"])
)

## 1.32. Inspect One-File DICOM Series

Some selected MRI image IDs contain only one DICOM file in the downloaded zip files. This may indicate an incomplete series, but it may also be a valid enhanced or multiframe DICOM where the whole MRI volume is stored in one file.

To distinguish these cases, this step inspects the internal zip path and uncompressed file size for each one-file series. Large files are more likely to represent complete MRI volumes; very small files would require further investigation.

In [ ]:
one_dcm_ids = set(one_dcm_series["image_id"].astype(int))

one_dcm_file_details = []

for zip_path in zip_paths:
    with zipfile.ZipFile(zip_path, "r") as z:
        for info in z.infolist():
            name = info.filename

            if not name.lower().endswith(".dcm"):
                continue

            matches = re.findall(r"I(\d+)", name)

            if len(matches) == 0:
                continue

            image_id = int(matches[-1])

            if image_id in one_dcm_ids:
                one_dcm_file_details.append({
                    "zip_file": zip_path.name,
                    "image_id": image_id,
                    "internal_path": name,
                    "uncompressed_size_mb": info.file_size / (1024 ** 2),
                    "compressed_size_mb": info.compress_size / (1024 ** 2)
                })

one_dcm_file_details = pd.DataFrame(one_dcm_file_details)

print("One-file DICOM details found:", len(one_dcm_file_details))

print("\nUncompressed file size summary:")
display(one_dcm_file_details["uncompressed_size_mb"].describe())

print("\nFiles smaller than 5 MB:")
small_one_file_dicoms = one_dcm_file_details[
    one_dcm_file_details["uncompressed_size_mb"] < 5
].copy()

print(len(small_one_file_dicoms))

display(
    one_dcm_file_details
    .merge(
        one_dcm_series[
            [
                "RID",
                "PTID",
                "subject_id",
                "final_group",
                "image_id",
                "description",
                "visit",
                "phase"
            ]
        ],
        on="image_id",
        how="left"
    )
    .sort_values("uncompressed_size_mb")
)

## 1.33. Interpretation of One-File DICOM Series

The selected 90-day MRI cohort includes 27 image IDs with only one DICOM file. These were inspected because a typical T1 MRI series often contains many slice-level DICOM files.

The one-file series are not very small files: their uncompressed sizes range from approximately 21 MB to 27 MB. This suggests that they may be valid single-file or enhanced/multiframe DICOM volumes rather than incomplete downloads.

These records should not be removed at the manifest stage. They will be kept in the cohort and checked again during DICOM-to-NIfTI conversion.

In [ ]:
DICOM_SERIES_QC_REPORT_PATH = QC_DIR / "clean_90d_dicom_series_qc_report.csv"
ONE_FILE_DICOM_DETAILS_PATH = QC_DIR / "clean_90d_one_file_dicom_details.csv"

# Full DICOM count QC report for all selected images
dicom_series_qc_report = clean_90d_extraction_manifest[
    [
        "RID",
        "PTID",
        "subject_id",
        "final_group",
        "image_id",
        "source_zip",
        "total_dcm_file_count",
        "description",
        "visit",
        "phase",
        "baseline_date",
        "study_date",
        "days_from_baseline_mri",
        "abs_days_from_baseline_mri"
    ]
].copy()

def dicom_count_status(count):
    if count == 1:
        return "single_large_dicom_check_at_conversion"
    elif count < 100:
        return "low_dicom_count_check"
    else:
        return "typical_dicom_series_count"

dicom_series_qc_report["dicom_count_qc_status"] = dicom_series_qc_report[
    "total_dcm_file_count"
].apply(dicom_count_status)

# Save reports
dicom_series_qc_report.to_csv(DICOM_SERIES_QC_REPORT_PATH, index=False)
one_dcm_file_details.to_csv(ONE_FILE_DICOM_DETAILS_PATH, index=False)

print("Saved DICOM series QC report:")
print(DICOM_SERIES_QC_REPORT_PATH)

print("\nSaved one-file DICOM details:")
print(ONE_FILE_DICOM_DETAILS_PATH)

print("\nDICOM count QC status:")
print(dicom_series_qc_report["dicom_count_qc_status"].value_counts())

print("\nOne-file DICOM size range:")
print("Min MB:", one_dcm_file_details["uncompressed_size_mb"].min())
print("Max MB:", one_dcm_file_details["uncompressed_size_mb"].max())